# CSE 251B Blended-Data Pipeline

This notebook prepares blended train shards, trains the nano model with root `val.bin` validation, runs the TA evaluation script, and packages the submission files under `results/<timestamp>` before copying them to Drive.

In [1]:
!nvidia-smi

Wed May 27 01:47:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P0             44W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 1. Mount Drive And Paths

In [2]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import shutil
import subprocess
import time

DRIVE_ROOT = Path('/content/drive/MyDrive/251B')
DRIVE_DATASET_DIR = DRIVE_ROOT / 'dataset' / 'blend'
DRIVE_RESULTS_DIR = DRIVE_ROOT / 'results'

REPO_DIR = Path('/content/milestone')
LOCAL_DATASET_DIR = REPO_DIR / 'data' / 'blend'

RUN_NAME = 'blend_nano_' + time.strftime('%Y%m%d_%H%M%S')
LOCAL_RESULTS_DIR = REPO_DIR / 'results' / RUN_NAME
DRIVE_RUN_DIR = DRIVE_RESULTS_DIR / RUN_NAME

DRIVE_DATASET_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('Drive dataset:', DRIVE_DATASET_DIR)
print('Drive results:', DRIVE_RESULTS_DIR)
print('Repo:', REPO_DIR)
print('Local dataset:', LOCAL_DATASET_DIR)
print('Local run results:', LOCAL_RESULTS_DIR)
print('Drive run results:', DRIVE_RUN_DIR)

Mounted at /content/drive
Drive dataset: /content/drive/MyDrive/251B/dataset/blend
Drive results: /content/drive/MyDrive/251B/results
Repo: /content/milestone
Local dataset: /content/milestone/data/blend
Local run results: /content/milestone/results/blend_nano_20260527_014738
Drive run results: /content/drive/MyDrive/251B/results/blend_nano_20260527_014738


## 2. Prepare Repository

In [3]:
!rm -rf /content/milestone
!git clone -b dataset-improved https://github.com/FufenNan/milestone.git /content/milestone
%cd /content/milestone
!git status --short --branch

LOCAL_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

Cloning into '/content/milestone'...
remote: Enumerating objects: 813, done.
remote: Counting objects: 100% (813/813), done.
remote: Compressing objects: 100% (378/378), done.
remote: Total 813 (delta 454), reused 776 (delta 428), pack-reused 0 (from 0)
Receiving objects: 100% (813/813), 7.76 MiB | 20.91 MiB/s, done.
Resolving deltas: 100% (454/454), done.
/content/milestone
## dataset-improved...origin/dataset-improved


## 3. Install Dependencies

In [4]:
!pip install -q datasets tiktoken tqdm

In [8]:
!pip install -q "datasets==2.19.2"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.1/542.1 kB 9.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 6.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.3.1 which is incompatible.


## 4. Copy Or Prepare Training Blend

In [5]:
DATASET_LOG = LOCAL_RESULTS_DIR / 'dataset_prepare.log'

def log(msg):
    line = f'[{time.strftime("%Y-%m-%d %H:%M:%S")}] {msg}'
    print(line)
    with open(DATASET_LOG, 'a', encoding='utf-8') as f:
        f.write(line + '\n')

BLEND_SOURCES = {
    'fineweb': {'dir': 'fineweb_edu', 'prefix': 'fineweb', 'min_shards': 20},
    'wikipedia': {'dir': 'wikipedia', 'prefix': 'wikipedia', 'min_shards': 9},
    'arxiv': {'dir': 'papers_arxiv', 'prefix': 'papers_arxiv', 'min_shards': 5},
    'pubmed': {'dir': 'papers_pubmed', 'prefix': 'papers_pubmed', 'min_shards': 2},
    'books': {'dir': 'books', 'prefix': 'books', 'min_shards': 7},
}

def count_shards(path):
    path = Path(path)
    return {
        name: len(sorted((path / spec['dir']).glob(f"{spec['prefix']}_train_*.bin")))
        for name, spec in BLEND_SOURCES.items()
    }

def format_counts(counts):
    return ', '.join(f'{name}={counts.get(name, 0)}' for name in BLEND_SOURCES)

def shard_bytes(path):
    path = Path(path)
    return sum(p.stat().st_size for p in path.rglob('*_train_*.bin'))

def has_ready_dataset(path):
    counts = count_shards(path)
    return all(counts[name] >= spec['min_shards'] for name, spec in BLEND_SOURCES.items())

def sync_dir(src, dst):
    src = Path(src)
    dst = Path(dst)
    dst.mkdir(parents=True, exist_ok=True)
    try:
        subprocess.run(['rsync', '-ah', '--info=progress2', f'{src}/', f'{dst}/'], check=True)
    except FileNotFoundError:
        shutil.copytree(src, dst, dirs_exist_ok=True)

SHARD_SIZE = 100_000_000
NUM_PROC = max(1, (os.cpu_count() or 2) // 2)

drive_counts = count_shards(DRIVE_DATASET_DIR)
log(f'Drive shards: {format_counts(drive_counts)}, size={shard_bytes(DRIVE_DATASET_DIR) / 1e9:.2f} GB')

if has_ready_dataset(DRIVE_DATASET_DIR):
    log('Copying blended shards from Drive to repo data/blend...')
    sync_dir(DRIVE_DATASET_DIR, LOCAL_DATASET_DIR)
else:
    log('Drive dataset is incomplete. Preparing blended shards under repo data/blend...')
    LOCAL_DATASET_DIR.mkdir(parents=True, exist_ok=True)
    for source_name, source_spec in BLEND_SOURCES.items():
        cmd = [
            'python', '-u', 'data/blend.py',
            '--data-root', str(LOCAL_DATASET_DIR),
            '--sources', source_name,
            '--streaming',
            '--shard-size', str(SHARD_SIZE),
            '--max-shards-per-source', str(source_spec['min_shards']),
            '--num-proc', str(NUM_PROC),
        ]
        log('Running: ' + ' '.join(cmd))
        with open(DATASET_LOG, 'a', encoding='utf-8') as log_file:
            proc = subprocess.Popen(cmd, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
            for line in proc.stdout:
                print(line, end='')
                log_file.write(line)
                log_file.flush()
            ret = proc.wait()
        if ret != 0:
            raise RuntimeError(f'Dataset preparation for {source_name} failed with exit code {ret}. See {DATASET_LOG}')
        log(f'Copying prepared {source_name} shards back to Drive...')
        sync_dir(LOCAL_DATASET_DIR / source_spec['dir'], DRIVE_DATASET_DIR / source_spec['dir'])
    log('Copying prepared blended shards back to Drive...')
    sync_dir(LOCAL_DATASET_DIR, DRIVE_DATASET_DIR)

local_counts = count_shards(LOCAL_DATASET_DIR)
log(f'Local shards: {format_counts(local_counts)}, size={shard_bytes(LOCAL_DATASET_DIR) / 1e9:.2f} GB')
assert has_ready_dataset(LOCAL_DATASET_DIR), 'Blended dataset is not ready in repo data/blend.'

[2026-05-27 01:48:17] Drive shards: fineweb=20, wikipedia=9, arxiv=5, pubmed=2, books=7, size=8.60 GB
[2026-05-27 01:48:17] Copying blended shards from Drive to repo data/blend...
[2026-05-27 01:55:31] Local shards: fineweb=20, wikipedia=9, arxiv=5, pubmed=2, books=7, size=8.60 GB


## 5. Train

In [6]:
RUN_INITIAL_TRAINING = True

if RUN_INITIAL_TRAINING:
    TRAIN_STDOUT = LOCAL_RESULTS_DIR / 'train_stdout.log'

    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'

    import importlib.util
    spec = importlib.util.spec_from_file_location('train_config', REPO_DIR / 'config' / 'config.py')
    train_config = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(train_config)
    print('Optimizer:', getattr(train_config, 'optimizer', 'adamw'))
    print('Muon lr:', getattr(train_config, 'muon_lr', None))

    cmd = ['python', '-u', 'train.py', '--config', 'config/config.py']
    # cmd = ['python', '-u', 'train.py', '--config', 'config/config_10000.py']
    print('Run name:', RUN_NAME)
    print('Command:', ' '.join(cmd))

    with open(TRAIN_STDOUT, 'w', encoding='utf-8') as log_file:
        proc = subprocess.Popen(cmd, cwd=REPO_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            print(line, end='')
            log_file.write(line)
            log_file.flush()
        ret = proc.wait()

    if ret != 0:
        raise RuntimeError(f'Training failed with exit code {ret}. See {TRAIN_STDOUT}')
else:
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    print('Skipping initial training; set RUN_INITIAL_TRAINING=True to run it.')


Optimizer: muon
Muon lr: 0.02
Run name: blend_nano_20260527_014738
Command: python -u train.py --config config/config.py
device: cuda
gradient accumulation steps: 33
train data mix per optimizer step: fineweb_edu=16, wikipedia=7, papers=5, books=5
validation data: /content/milestone/val.bin
parameters: 99,027,200
optimizer: muon | muon tensors: 60 (66,846,720 params) | adamw fallback tensors: 26 (32,180,480 params)
/usr/local/lib/python3.12/dist-packages/torch/_inductor/lowering.py:2212: UserWarning: Torchinductor does not support code generation for complex operators. Performance may be worse than eager.
  warnings.warn(
step     0 | val loss 10.9218
step     0 | train loss 10.943378 | lr 7.5000e-07 | norm 8.2058 | 9100 tok/s
step    10 | train loss 10.603416 | lr 8.2500e-06 | norm 7.3749 | 182343 tok/s
step    20 | train loss 9.976182 | lr 1.5750e-05 | norm 5.2969 | 182206 tok/s
step    30 | train loss 9.351460 | lr 2.3250e-05 | norm 3.0908 | 182000 tok/s
step    40 | train loss 9.03

## 5b. Continue Pre-Training From Drive Checkpoint


In [ ]:
CONTINUE_PRETRAINING = False
RESUME_RUN_NAME = 'previous_run_name'
RESUME_CHECKPOINT_DRIVE = DRIVE_ROOT / 'results' / RESUME_RUN_NAME / 'resume_checkpoint.pt'
LEGACY_RESUME_CHECKPOINT_DRIVE = DRIVE_ROOT / 'results' / RESUME_RUN_NAME / 'checkpoint.pt'
CONTINUE_STEPS = 10000

if CONTINUE_PRETRAINING:
    if not RESUME_CHECKPOINT_DRIVE.exists() and LEGACY_RESUME_CHECKPOINT_DRIVE.exists():
        RESUME_CHECKPOINT_DRIVE = LEGACY_RESUME_CHECKPOINT_DRIVE
    local_resume = REPO_DIR / 'checkpoints' / 'checkpoint.pt'
    assert RESUME_CHECKPOINT_DRIVE.exists(), f'Missing resume checkpoint: {RESUME_CHECKPOINT_DRIVE}'
    local_resume.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(RESUME_CHECKPOINT_DRIVE, local_resume)

    CONTINUE_STDOUT = LOCAL_RESULTS_DIR / 'continue_train_stdout.log'
    cmd = [
        'python', '-u', 'train.py',
        '--config', 'config/config.py',
        '--resume', 'checkpoints/checkpoint.pt',
        '--steps', str(CONTINUE_STEPS),
    ]
    print('Continue command:', ' '.join(cmd))

    with open(CONTINUE_STDOUT, 'w', encoding='utf-8') as log_file:
        proc = subprocess.Popen(
            cmd,
            cwd=REPO_DIR,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        for line in proc.stdout:
            print(line, end='')
            log_file.write(line)
            log_file.flush()
        ret = proc.wait()

    if ret != 0:
        raise RuntimeError(f'Continue pre-training failed with exit code {ret}. See {CONTINUE_STDOUT}')
else:
    print('Skipping continue pre-training; set CONTINUE_PRETRAINING=True to resume from Drive checkpoint.')


## 6. Run TA Evaluation

In [7]:
LOCAL_CHECKPOINT = REPO_DIR / 'checkpoints' / 'checkpoint.pt'
EVAL_DATA = REPO_DIR / 'val.bin'
TA_EVAL_JSON = LOCAL_RESULTS_DIR / 'eval_val.json'

assert LOCAL_CHECKPOINT.exists(), f'Missing checkpoint: {LOCAL_CHECKPOINT}'
assert EVAL_DATA.exists(), f'Missing eval data: {EVAL_DATA}'

cmd = [
    'python', '-u', 'evaluate.py',
    '--model_dir', str(REPO_DIR),
    '--checkpoint_filename', 'checkpoints/checkpoint.pt',
    '--data', str(EVAL_DATA),
    '--device', 'cuda',
    '--batch_size', '1',
    '--output_json', str(TA_EVAL_JSON),
]
print('Command:', ' '.join(cmd))

proc = subprocess.Popen(cmd, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
ret = proc.wait()

if ret != 0:
    raise RuntimeError(f'TA evaluation failed with exit code {ret}.')

Command: python -u evaluate.py --model_dir /content/milestone --checkpoint_filename checkpoints/checkpoint.pt --data /content/milestone/val.bin --device cuda --batch_size 1 --output_json /content/milestone/results/blend_nano_20260527_014738/eval_val.json

--- Evaluating local model: /content/milestone ---
Loading model from /content/milestone/checkpoints/checkpoint.pt...
Total parameters:       99,027,200
Trainable parameters:   99,027,200
Evaluating on /content/milestone/val.bin (block_size=1024)...

  Perplexity:          25.4062
  Avg loss (nats):     3.234992
  Tokens evaluated:    5,170,176
  Total parameters:    99,027,200
  Eval time:           83.6s

Results written to /content/milestone/results/blend_nano_20260527_014738/eval_val.json


## 7. Package Results And Copy To Drive

In [8]:
import torch

FULL_CHECKPOINT = REPO_DIR / 'checkpoints' / 'checkpoint.pt'
BEST_FULL_CHECKPOINT = REPO_DIR / 'checkpoints' / 'best_checkpoint.pt'
SUBMISSION_CHECKPOINT = LOCAL_RESULTS_DIR / 'checkpoint.pt'
BEST_SUBMISSION_CHECKPOINT = LOCAL_RESULTS_DIR / 'best_checkpoint.pt'

def export_model_only_checkpoint(src, dst):
    checkpoint = torch.load(src, map_location='cpu', weights_only=False)
    if isinstance(checkpoint, dict) and 'model' in checkpoint:
        payload = {
            'model': checkpoint['model'],
            'model_config': checkpoint.get('model_config'),
            'global_step': checkpoint.get('global_step'),
            'val_loss': checkpoint.get('val_loss'),
            'best_val_loss': checkpoint.get('best_val_loss'),
        }
        torch.save(payload, dst)
    else:
        shutil.copy2(src, dst)

assert FULL_CHECKPOINT.exists(), f'Missing expected file: {FULL_CHECKPOINT}'
export_model_only_checkpoint(FULL_CHECKPOINT, SUBMISSION_CHECKPOINT)
if BEST_FULL_CHECKPOINT.exists():
    export_model_only_checkpoint(BEST_FULL_CHECKPOINT, BEST_SUBMISSION_CHECKPOINT)

submission_files = {
    REPO_DIR / 'config' / 'config.py': LOCAL_RESULTS_DIR / 'config.py',
    # REPO_DIR / 'config' / 'config_10000.py': LOCAL_RESULTS_DIR / 'config.py',
    REPO_DIR / 'model.py': LOCAL_RESULTS_DIR / 'model.py',
}

optional_logs = [
    REPO_DIR / 'checkpoints' / 'train.log',
    REPO_DIR / 'results' / 'train.log',
    REPO_DIR / 'checkpoints' / 'checkpoint_metadata.pt',
    REPO_DIR / 'checkpoints' / 'best_checkpoint_metadata.pt',
]

resume_checkpoints = {
    FULL_CHECKPOINT: LOCAL_RESULTS_DIR / 'resume_checkpoint.pt',
    BEST_FULL_CHECKPOINT: LOCAL_RESULTS_DIR / 'best_resume_checkpoint.pt',
}

for src, dst in submission_files.items():
    assert src.exists(), f'Missing expected file: {src}'
    shutil.copy2(src, dst)

for src in optional_logs:
    if src.exists():
        shutil.copy2(src, LOCAL_RESULTS_DIR / src.name)

for src, dst in resume_checkpoints.items():
    if src.exists():
        shutil.copy2(src, dst)

DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=True)
sync_dir(LOCAL_RESULTS_DIR, DRIVE_RUN_DIR)

print('Local results:', LOCAL_RESULTS_DIR)
print('Drive results:', DRIVE_RUN_DIR)
print('Packaged files:')
for path in sorted(LOCAL_RESULTS_DIR.iterdir()):
    print(' -', path.name)

Local results: /content/milestone/results/blend_nano_20260527_014738
Drive results: /content/drive/MyDrive/251B/results/blend_nano_20260527_014738
Packaged files:
 - checkpoint.pt
 - checkpoint_metadata.pt
 - config.py
 - dataset_prepare.log
 - eval_val.json
 - model.py
 - train.log
 - train_stdout.log


# 8. Exit Colab

In [9]:
from google.colab import runtime
runtime.unassign()